# Day 13 · Exercise 4: Retrieve Relevant Context

**What you'll build:** a function that embeds a question, queries a ChromaDB collection, and returns the text of the top-k most semantically similar chunks.

**Why it matters:** retrieval quality determines answer quality — if the wrong chunks come back, even a perfect LLM will produce a wrong answer.

## Your Implementation

In [ ]:
import chromadb
import ollama


def retrieve_context(
    question: str,
    collection,
    top_k: int = 3,
) -> list[str]:
    """Embed a question and retrieve the top-k most relevant chunks.

    Args:
        question: The user's question to search for.
        collection: A ChromaDB Collection already populated with embedded chunks.
        top_k: Number of most-similar chunks to return.

    Returns:
        List of up to top_k chunk text strings, most relevant first.

    Example:
        >>> # (assumes collection already has indexed documents)
        >>> chunks = retrieve_context("What is Python?", collection, top_k=2)
        >>> isinstance(chunks, list)
        True
        >>> all(isinstance(c, str) for c in chunks)
        True
    """
    # ── YOUR CODE HERE ─────────────────────────────────────────
    pass
    # ───────────────────────────────────────────────────────────

## Check Your Work
Run the cell below — it runs 4 automated checks and shows ✅ / ❌ for each.

In [ ]:
_PASS, _FAIL = '✅', '❌'

def _run_checks():
    score = 0
    total = 4

    # Setup: build a tiny collection with 3 pre-embedded chunks
    try:
        import chromadb as _chromadb
        _client = _chromadb.Client()
        _coll = _client.get_or_create_collection("ex04_test")
        _chunks = [
            "Python is a high-level programming language.",
            "Machine learning trains models on data.",
            "ChromaDB stores vector embeddings for retrieval.",
        ]
        for _i, _chunk in enumerate(_chunks):
            _emb = ollama.embeddings(model="nomic-embed-text", prompt=_chunk)
            _coll.add(
                ids=[f"chunk_{_i:04d}"],
                embeddings=[_emb["embedding"]],
                documents=[_chunk],
                metadatas=[{"source": "test", "chunk_index": _i}],
            )
    except Exception as e:
        print(f"{_FAIL} Setup: could not build test collection: {e}")
        return

    # Check 1: returns a list
    try:
        result = retrieve_context("What is Python?", _coll, top_k=2)
        if not isinstance(result, list):
            print(f"{_FAIL} Check 1: retrieve_context should return a list, got {type(result).__name__}")
            return
        print(f"{_PASS} Check 1: retrieve_context returns a list")
        score += 1
    except Exception as e:
        print(f"{_FAIL} Check 1: raised {type(e).__name__}: {e}")
        return

    # Check 2: returns strings
    if all(isinstance(c, str) for c in result):
        print(f"{_PASS} Check 2: all returned items are strings")
        score += 1
    else:
        print(f"{_FAIL} Check 2: expected list of strings, got mixed types")
        return

    # Check 3: honours top_k
    if len(result) <= 2:
        print(f"{_PASS} Check 3: returned {len(result)} chunk(s) (≤ top_k=2)")
        score += 1
    else:
        print(f"{_FAIL} Check 3: top_k=2 but returned {len(result)} chunks")

    # Check 4: most relevant chunk is about Python (semantic check)
    if result and "Python" in result[0]:
        print(f"{_PASS} Check 4: top result is semantically relevant (contains 'Python')")
        score += 1
    else:
        top = result[0] if result else "(empty)"
        print(f"{_FAIL} Check 4: expected top result about Python, got: '{top[:60]}'")

    if score == total:
        print("🎉 Exercise complete!")
    print(f"  {score}/{total} passed.")

_run_checks()

## Bonus Challenge
Add a `min_score` parameter to `retrieve_context` that filters out chunks with a distance score above the threshold. (Hint: `collection.query()` also returns `result['distances'][0]` — a list of float distances where smaller = more similar in ChromaDB's default L2 metric.) This foreshadows the relevance filtering you'll implement on Day 14.

## Solution

<details>
<summary>Click to reveal — try on your own first</summary>

```python
def retrieve_context(
    question: str,
    collection,
    top_k: int = 3,
) -> list[str]:
    q_emb = ollama.embeddings(model="nomic-embed-text", prompt=question)
    results = collection.query(
        query_embeddings=[q_emb["embedding"]],
        n_results=top_k,
    )
    return results["documents"][0]
```

**Why this works:** `collection.query()` takes a batch of query embeddings — `[q_emb["embedding"]]` wraps the single vector in a list. The return value has the same batch structure: `result["documents"][0]` unwraps that outer list to get the texts for your one query. ChromaDB handles the nearest-neighbour search internally.
</details>